In [2]:
import ssl
import nltk
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lap15114/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
from context_cite import ContextCiter

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lap15114/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
from dotenv import load_dotenv
import os
import requests

load_dotenv(dotenv_path="../.env")

API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = "https://chat.zingplay.com/api/chat/completions"

In [11]:
context = "The sky is blue because of Rayleigh scattering."
question = "Why is the sky blue?"
answer = "The sky is blue because of Rayleigh scattering of sunlight."

response = requests.post(
    BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "local-model-mini",
        "messages": [
            {"role": "system", "content": "You answer in one short sentence."},
            {"role": "user", "content": f"Context: {context}\n\n{question}"},
            {"role": "assistant", "content": answer}
        ],
        "temperature": 0,
        "max_tokens": 1,
        "prompt_logprobs": 1
    }
)

print(response.json())

{'id': 'chatcmpl-92267c136ff04a97', 'created': 1781084811, 'model': 'local-model-mini', 'object': 'chat.completion', 'choices': [{'finish_reason': 'length', 'index': 0, 'message': {'role': 'assistant', 'reasoning_content': 'Thinking', 'provider_specific_fields': {'refusal': None, 'reasoning': 'Thinking', 'reasoning_content': 'Thinking'}, 'content': None}, 'provider_specific_fields': {'token_ids': None, 'stop_reason': None}}], 'usage': {'completion_tokens': 1, 'prompt_tokens': 62, 'total_tokens': 63}, 'prompt_logprobs': [None, {'8678': {'logprob': -1.9102483987808228, 'rank': 2, 'decoded_token': 'system'}, '846': {'logprob': -0.16024836897850037, 'rank': 1, 'decoded_token': 'user'}}, {'198': {'logprob': -5.2689116273541003e-05, 'rank': 1, 'decoded_token': '\n'}}, {'2523': {'logprob': -2.0911359786987305, 'rank': 1, 'decoded_token': 'You'}}, {'4087': {'logprob': -10.956963539123535, 'rank': 46, 'decoded_token': ' answer'}, '513': {'logprob': -0.2069634050130844, 'rank': 1, 'decoded_token

In [12]:
logprobs = response.json()["prompt_logprobs"]

# The answer tokens are at the end - extract the top-ranked logprob for each position
answer_logprobs = []
for token_dict in logprobs:
    if token_dict is None:
        continue
    # Get the logprob of whichever token actually appeared (rank 1)
    top = min(token_dict.values(), key=lambda x: x["rank"])
    answer_logprobs.append((top["decoded_token"], top["logprob"]))

# Print all tokens so we can see the full sequence
for token, lp in answer_logprobs:
    print(f"{repr(token):30s} {lp:.4f}")

'user'                         -0.1602
'\n'                           -0.0001
'You'                          -2.0911
' are'                         -0.2070
' questions'                   -1.2731
' a'                           -1.1405
' word'                        -1.1104
' sentence'                    -0.6096
'.'                            -0.3293
'<|im_end|>'                   -0.6460
'\n'                           -0.0000
'<|im_start|>'                 -0.0000
'assistant'                    -0.0164
'\n'                           -0.0001
'What'                         -1.4875
':'                            -0.1513
'\n'                           -0.9574
' following'                   -3.6081
' is'                          -0.2606
' blue'                        -0.5899
'.'                            -0.5762
' of'                          -0.7806
' Ray'                         -0.7562
'leigh'                        -0.0091
' scattering'                  -0.0268
'.'                      

In [13]:
import context_cite
print(dir(context_cite))

['ContextCiter', 'VERSION', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'context_citer', 'context_partitioner', 'solver', 'utils']


In [14]:
print(dir(context_cite.solver))
print(dir(context_cite.context_partitioner))
print(dir(context_cite.utils))

['ABC', 'BaseSolver', 'Lasso', 'LassoRegression', 'NDArray', 'StandardScaler', 'Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'abstractmethod', 'make_pipeline', 'np']
['ABC', 'BaseContextPartitioner', 'List', 'NDArray', 'Optional', 'SimpleContextPartitioner', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'abstractmethod', 'np', 'split_text']
['Any', 'DataCollatorForSeq2Seq', 'DataLoader', 'Dataset', 'English', 'List', 'NDArray', 'Optional', 'Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_apply_color_scale', '_color_scale', '_compute_logit_probs', '_create_mask', '_create_regression_dataset', '_get_response_logit_probs', '_make_loader', 'aggregate_logit_probs', 'ch', 'char_to_token', 'get_attributions_df', 'get_masks_and_logit_probs', 'highlight_word_indices', 'nltk', 'np', 'pd', 'split_text',

In [15]:
from context_cite.context_partitioner import SimpleContextPartitioner

context = """The meal cap for domestic travel is $50 per day. International travel requires manager approval. Hotel expenses are reimbursed up to $200 per night."""

partitioner = SimpleContextPartitioner(context, source_type="sentence")
print(partitioner.sources)

['The meal cap for domestic travel is $50 per day.', 'International travel requires manager approval.', 'Hotel expenses are reimbursed up to $200 per night.']


In [16]:
print(partitioner.num_sources)
print(partitioner.sources)

# See what a masked context looks like
import numpy as np

# mask out the first chunk
mask = np.array([False, True, True])
print(partitioner.get_context(mask))

3
['The meal cap for domestic travel is $50 per day.', 'International travel requires manager approval.', 'Hotel expenses are reimbursed up to $200 per night.']
International travel requires manager approval. Hotel expenses are reimbursed up to $200 per night.


In [17]:
from context_cite.solver import LassoRegression

# Fake ablation masks - 5 ablations, 3 chunks
masks = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 1],
    [0, 0, 1],
])

# Fake scores - how likely R was given each masked context
scores = np.array([0.8, 0.3, 0.2, 0.9, 0.1])

solver = LassoRegression()
weights, bias = solver.fit(masks, scores, num_output_tokens=1)
print(weights)

[ 0.43550589  0.33550861 -0.01449275]
